# Weather Forecasting using ConvLSTM

Portfolio notebook for next-frame spatiotemporal weather-grid forecasting. The original executed notebook is preserved in `archive/`.

## Responsible Use
This educational synthetic-data project is not an official or safety-critical weather forecasting system.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

## Load Safe Sample Data

In [ ]:
payload = np.load(PROJECT_ROOT / "data" / "sample_weather_sequences.npz")
X, y, future_y = payload["X"], payload["y"], payload["future_y"]
X.shape, y.shape, future_y.shape

## Inspect the Input Sequence

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(14, 3))
for i, ax in enumerate(axes):
    ax.imshow(X[0, i, :, :, 0], vmin=0, vmax=1)
    ax.set_title(f"Frame {i+1}")
    ax.axis("off")
plt.tight_layout()

## Build the ConvLSTM Architecture

In [ ]:
from src.model_training import build_convlstm
model = build_convlstm((6, 24, 24, 1))
model.summary()

## Load the Supplied Pretrained Model

In [ ]:
import tensorflow as tf
model = tf.keras.models.load_model(PROJECT_ROOT / "models" / "convlstm_weather_forecast.keras", compile=False)

## Predict the Next Weather Map

In [ ]:
prediction = model.predict(X[:1], verbose=0)[0]
prediction.shape

## Evaluate and Visualize

In [ ]:
from src.model_evaluation import evaluate_weather_map
from src.visualization import comparison_figure, error_heatmap_figure

metrics = evaluate_weather_map(y[0], prediction, threshold=0.5)
metrics

In [ ]:
comparison_figure(X[0, -1], y[0], prediction)

In [ ]:
error_heatmap_figure(y[0], prediction)

## Recursive Forecasting

In [ ]:
from src.forecasting_pipeline import recursive_forecast
metadata = json.loads((PROJECT_ROOT / "models" / "model_metadata.json").read_text())
forecasts = recursive_forecast(model, X[0], metadata, steps=4)
forecasts.shape

## Recorded Test Comparison

In [ ]:
pd.read_csv(PROJECT_ROOT / "outputs" / "baseline_comparison.csv")

## Next Steps
Use licensed real weather grids, chronological backtesting, uncertainty estimation, multi-output training, and stronger weather-specific metrics before considering operational use.